In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip uninstall -y numpy peft accelerate transformers datasets -q

!pip install -q \
  numpy==1.26.4 \
  torch \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  peft==0.7.1 \
  safetensors==0.4.2 \
  scikit-learn \
  sentencepiece

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but 

In [ ]:
import os
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    CLIPProcessor,
    CLIPModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from transformers.modeling_outputs import SequenceClassifierOutput
from peft import LoraConfig, get_peft_model
from torch import nn
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda


In [ ]:
dataset = load_dataset("anson-huang/mirage-news")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_name)


class CLIPForMFND(nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()

        base_model = CLIPModel.from_pretrained(model_name)

        # 🔥 LoRA config đúng cho CLIP attention
        lora_config = LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=[
                "q_proj",
                "v_proj"
            ],
            lora_dropout=0.1,
            bias="none"
        )

        self.clip = get_peft_model(base_model, lora_config)

        self.classifier = nn.Linear(
            base_model.config.projection_dim * 2,
            num_labels
        )

        print("\nTrainable parameters:")
        self.clip.print_trainable_parameters()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        pixel_values=None,
        labels=None,
    ):
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            return_dict=True
        )

        text_embeds = outputs.text_embeds
        image_embeds = outputs.image_embeds

        fused = torch.cat([text_embeds, image_embeds], dim=1)
        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)


model = CLIPForMFND(model_name).to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



Trainable parameters:
trainable params: 491,520 || all params: 151,768,833 || trainable%: 0.3238609603066527


In [ ]:
def preprocess(example):
    inputs = processor(
        text=example["text"],
        images=example["image"],
        truncation=True,
        padding="max_length",
        max_length=77,
        return_tensors="pt"
    )

    return {
        "input_ids": inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "pixel_values": inputs["pixel_values"][0],
        "labels": torch.tensor(example["label"], dtype=torch.long)
    }


encoded_dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names
)

encoded_dataset.set_format("torch")


Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    acc = accuracy_score(labels, preds)
    auc = roc_auc_score(labels, probs[:, 1])

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }


In [ ]:
output_dir = "/content/drive/MyDrive/CLIP_LoRA_MirageNews"
os.makedirs(output_dir, exist_ok=True)


training_args = TrainingArguments(
    output_dir=output_dir,

    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.01,

    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    fp16=torch.cuda.is_available(),
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.644000,0.397645,0.988000,0.981073,0.995200,0.988086,0.997801
2,0.369600,0.240031,0.995600,0.995997,0.995200,0.995598,0.999431
3,0.250200,0.183305,0.995200,0.992051,0.998400,0.995215,0.999627
4,0.170300,0.158899,0.996800,0.995215,0.998400,0.996805,0.999748
5,0.158700,0.151778,0.996800,0.995215,0.998400,0.996805,0.999752


TrainOutput(global_step=3125, training_loss=0.2926027185058594, metrics={'train_runtime': 1258.3787, 'train_samples_per_second': 39.734, 'train_steps_per_second': 2.483, 'total_flos': 0.0, 'train_loss': 0.2926027185058594, 'epoch': 5.0})

In [ ]:
metrics = trainer.evaluate()
print(metrics)


{'eval_loss': 0.15889932215213776, 'eval_accuracy': 0.9968, 'eval_precision': 0.9952153110047847, 'eval_recall': 0.9984, 'eval_f1': 0.9968051118210862, 'eval_auc': 0.99974848, 'eval_runtime': 34.858, 'eval_samples_per_second': 71.719, 'eval_steps_per_second': 2.266, 'epoch': 5.0}


In [ ]:
model.clip.save_pretrained(output_dir)
processor.save_pretrained(output_dir)

torch.save(
    model.classifier.state_dict(),
    os.path.join(output_dir, "classifier.pt")
)

print("Model saved to:", output_dir)


Model saved to: /content/drive/MyDrive/CLIP_LoRA_MirageNews


In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable:,}")
print(f"Total params: {total:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

Trainable params: 493,570
Total params: 151,770,883
Trainable %: 0.3252%


In [ ]:
# ─── 1. Install (giữ nguyên theo yêu cầu) ────────────────────────────────────
!pip uninstall -y numpy peft accelerate transformers datasets -q
!pip install -q \
  numpy==1.26.4 \
  torch \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  peft==0.7.1 \
  safetensors==0.4.2 \
  scikit-learn \
  sentencepiece

import os
os.kill(os.getpid(), 9)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but 

In [ ]:
# ─── 2. Setup ─────────────────────────────────────────────────────────────────
import os, time, functools
import torch
import numpy as np
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader
from transformers.modeling_outputs import SequenceClassifierOutput
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import CLIPProcessor, CLIPModel
from datasets import load_dataset

from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR   = "/content/drive/MyDrive/CLIP_LoRA_MirageNews"
model_name = "openai/clip-vit-base-patch32"
device     = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ─── 3. Model class ───────────────────────────────────────────────────────────
class CLIPForMFND(nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()
        base_model  = CLIPModel.from_pretrained(model_name)
        lora_config = LoraConfig(
            r=8, lora_alpha=16,
            target_modules=["q_proj", "v_proj"],
            lora_dropout=0.1, bias="none"
        )
        self.clip       = get_peft_model(base_model, lora_config)
        self.classifier = nn.Linear(base_model.config.projection_dim * 2, num_labels)

    def forward(self, input_ids=None, attention_mask=None,
                pixel_values=None, labels=None):
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            return_dict=True
        )
        fused  = torch.cat([outputs.text_embeds, outputs.image_embeds], dim=1)
        logits = self.classifier(fused)
        loss   = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits)

# ─── 4. Load model từ Drive ───────────────────────────────────────────────────
print("⏳ Loading CLIP ViT-B/32 LoRA...")
processor  = CLIPProcessor.from_pretrained(model_name)
base_model = CLIPModel.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1, bias="none"
)
peft_clip = get_peft_model(base_model, lora_config)
peft_clip.load_adapter(SAVE_DIR, adapter_name="default")

model = CLIPForMFND.__new__(CLIPForMFND)
nn.Module.__init__(model)
model.clip       = peft_clip
model.classifier = nn.Linear(base_model.config.projection_dim * 2, 2)

torch.load = functools.partial(torch.load, weights_only=False)
clf_state  = torch.load(os.path.join(SAVE_DIR, "classifier.pt"), map_location=device)
model.classifier.load_state_dict(clf_state)

model = model.to(device)
model.eval()
print("✅ Model loaded from Drive")

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.4f}%)")

# ─── 5. Preprocess + Evaluate helpers ────────────────────────────────────────
def make_test_dataset(split_name):
    ds = load_dataset("anson-huang/mirage-news", split=split_name)
    def preprocess(example):
        inputs = processor(
            text=example["text"], images=example["image"],
            truncation=True, padding="max_length",
            max_length=77, return_tensors="pt"
        )
        return {
            "input_ids":      inputs["input_ids"][0],
            "attention_mask": inputs["attention_mask"][0],
            "pixel_values":   inputs["pixel_values"][0],
            "labels":         example["label"]
        }
    encoded = ds.map(preprocess, remove_columns=ds.column_names, desc=split_name)
    return encoded

def collate_fn(batch):
    result = {}
    for k in batch[0]:
        arr = np.array([b[k] for b in batch])
        t   = torch.tensor(arr)
        if k == "pixel_values":
            t = t.float()
        result[k] = t
    return result

def evaluate_split(dataset):
    loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(
                input_ids      = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
                pixel_values   = batch["pixel_values"].to(device)
            )
            all_logits.append(out.logits.cpu())
            all_labels.append(batch["labels"])
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds  = np.argmax(probs, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": p, "recall": r, "f1": f1,
        "auc":       roc_auc_score(labels, probs[:, 1])
    }

# ─── 6. Test Evaluation ───────────────────────────────────────────────────────
test_splits = {
    "ID  test1_nyt_mj":    "test1_nyt_mj",
    "OOD test2_bbc_dalle": "test2_bbc_dalle",
    "OOD test3_cnn_dalle": "test3_cnn_dalle",
    "OOD test4_bbc_sdxl":  "test4_bbc_sdxl",
    "OOD test5_cnn_sdxl":  "test5_cnn_sdxl",
}

print("\n📊 Test Evaluation — CLIP ViT-B/32 LoRA (MIRAGE-News):")
print(f"{'Split':<26} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>8}")
print("-" * 63)

results, ood_names = {}, []

for name, split in test_splits.items():
    td            = make_test_dataset(split)
    m             = evaluate_split(td)
    results[name] = m
    print(f"{name:<26} {m['accuracy']*100:>6.2f}% {m['precision']*100:>6.2f}% "
          f"{m['recall']*100:>6.2f}% {m['f1']*100:>6.2f}% {m['auc']:>8.5f}")
    if name.startswith("OOD"):
        ood_names.append(name)

print("-" * 63)
ood_avg = {m: np.mean([results[n][m] for n in ood_names])
           for m in ["accuracy", "precision", "recall", "f1", "auc"]}
print(f"{'OOD Average':<26} {ood_avg['accuracy']*100:>6.2f}% {ood_avg['precision']*100:>6.2f}% "
      f"{ood_avg['recall']*100:>6.2f}% {ood_avg['f1']*100:>6.2f}% {ood_avg['auc']:>8.5f}")

# ─── 7. Latency + VRAM ───────────────────────────────────────────────────────
dummy_image = Image.new("RGB", (224, 224), (128, 128, 128))
dummy_text  = "This is a sample news headline for inference measurement."
inp = processor(text=dummy_text, images=dummy_image,
                truncation=True, padding="max_length",
                max_length=77, return_tensors="pt")
input_ids      = inp["input_ids"].to(device)
attention_mask = inp["attention_mask"].to(device)
pixel_values   = inp["pixel_values"].float().to(device)

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

with torch.no_grad():
    for _ in range(10):
        model(input_ids, attention_mask, pixel_values)

latencies = []
with torch.no_grad():
    for _ in range(100):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(input_ids, attention_mask, pixel_values)
        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

vram = torch.cuda.max_memory_allocated() / 1024**2 if device == "cuda" else 0
print(f"\n{'='*40}")
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Latency median  : {np.median(latencies):.2f} ms")
print(f"VRAM (peak)     : {vram:.1f} MB")
print(f"{'='*40}")

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_no

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cpu
⏳ Loading CLIP ViT-B/32 LoRA...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

OOD test3_cnn_dalle         66.00%  78.57%  44.00%  56.41%  0.84616


test4_bbc_sdxl:   0%|          | 0/500 [00:00<?, ? examples/s]

OOD test4_bbc_sdxl          79.20%  76.84%  83.60%  80.08%  0.90254


test5_cnn_sdxl:   0%|          | 0/500 [00:00<?, ? examples/s]

OOD test5_cnn_sdxl          84.20%  87.34%  80.00%  83.51%  0.94136
---------------------------------------------------------------
OOD Average                 74.20%  78.76%  66.00%  70.84%  0.87799

Total params    : 151,770,883
Trainable params: 493,570
Latency median  : 378.43 ms
VRAM (peak)     : 0.0 MB
